# Polymer Combined Supervisor

Canonical unified polymer notebook. Edit the config cell below for one-off runs, or change `systems/polymer/notebook_params.py` for repo-wide defaults.

## User Config

In [1]:
from pathlib import Path
import os

from systems.polymer import get_polymer_notebook_defaults
from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_grouped_notebook_summary

NB = get_polymer_notebook_defaults("combined")
RUN_MODE = NB["run_mode"]
STYLE_PROFILE = NB["style_profile"]
SAVE_PDF = NB["save_pdf"]
ENABLE_HORIZON = NB["enable_horizon"]
HORIZON_AGENT_KIND = str(NB.get("horizon_agent_kind", "dqn")).lower()
HORIZON_STATE_MODE = NB["horizon_state_mode"]
ENABLE_MATRIX = NB["enable_matrix"]
MATRIX_AGENT_KIND = NB["matrix_agent_kind"]
MATRIX_STATE_MODE = NB["matrix_state_mode"]
ENABLE_WEIGHTS = NB["enable_weights"]
WEIGHTS_AGENT_KIND = NB["weights_agent_kind"]
WEIGHTS_STATE_MODE = NB["weights_state_mode"]
ENABLE_RESIDUAL = NB["enable_residual"]
RESIDUAL_AGENT_KIND = NB["residual_agent_kind"]
RESIDUAL_STATE_MODE = NB["residual_state_mode"]
USE_RHO_AUTHORITY = NB["authority_use_rho"]
APPEND_RHO_TO_STATE = bool(NB["append_rho_to_state"])
AUTHORITY_BETA_RES = NB["authority_beta_res"]
AUTHORITY_DU0_RES = NB["authority_du0_res"]
AUTHORITY_ETA_TOL = NB["authority_eta_tol"]
AUTHORITY_RHO_FLOOR = NB["authority_rho_floor"]
AUTHORITY_RHO_POWER = NB["authority_rho_power"]
RHO_MAPPING_MODE = NB["rho_mapping_mode"]
AUTHORITY_RHO_K = NB["authority_rho_k"]
RESIDUAL_ZERO_DEADBAND_ENABLED = bool(NB["residual_zero_deadband_enabled"])
RESIDUAL_ZERO_TRACKING_RAW_THRESHOLD = NB["residual_zero_tracking_raw_threshold"]
RESIDUAL_ZERO_INNOVATION_RAW_THRESHOLD = NB["residual_zero_innovation_raw_threshold"]
POLYMER_DATA_DIR_OVERRIDE = NB["data_dir_override"]
POLYMER_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
RESULT_PREFIX_OVERRIDE = NB["result_prefix_override"]
COMPARE_PREFIX_OVERRIDE = NB["compare_prefix_override"]
BASELINE_MPC_PATH_OVERRIDE = NB["baseline_mpc_path_override"]
N_TESTS_OVERRIDE = NB["n_tests_override"]
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
WARM_START_OVERRIDE = NB["warm_start_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = NB["plot_start_episode_override"]
COMPARE_START_EPISODE_OVERRIDE = NB["compare_start_episode_override"]
REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(data_dir_override=POLYMER_DATA_DIR_OVERRIDE, results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE)
os.chdir(REPO_ROOT)
RUN_PROFILE = NB["run_profiles"][RUN_MODE]

def build_combined_suffix() -> str:
    parts = []
    if ENABLE_HORIZON:
        parts.append(f"h_{HORIZON_AGENT_KIND}_{HORIZON_STATE_MODE}")
    if ENABLE_MATRIX:
        parts.append(f"m_{MATRIX_AGENT_KIND}_{MATRIX_STATE_MODE}")
    if ENABLE_WEIGHTS:
        parts.append(f"w_{WEIGHTS_AGENT_KIND}_{WEIGHTS_STATE_MODE}")
    if ENABLE_RESIDUAL:
        rho_suffix = "rho" if USE_RHO_AUTHORITY else "no_rho"
        parts.append(f"r_{RESIDUAL_AGENT_KIND}_{RESIDUAL_STATE_MODE}_{rho_suffix}")
    return "__".join(parts) if parts else "baseline"


## Imports

In [2]:
import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from DuelingDQN.dueling_dqn_agent import DuelingDQNAgent
from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from TD3Agent.agent import TD3Agent
from systems.polymer import (
    HORIZON_CONTROL_GRID,
    HORIZON_PREDICT_GRID,
    POLYMER_DELTA_T_HOURS,
    POLYMER_DESIGN_PARAMS,
    POLYMER_INPUT_BOUNDS,
    POLYMER_OBSERVER_POLES,
    POLYMER_RL_SETPOINTS_PHYS,
    POLYMER_SETPOINT_RANGE_PHYS,
    POLYMER_SYSTEM_METADATA,
    POLYMER_SS_INPUTS,
    POLYMER_SYSTEM_PARAMS,
    RL_REWARD_DEFAULTS,
    load_polymer_system_data,
)
from utils.combined_runner import run_combined_supervisor
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.multiplier_sensitivity import (
    extract_suggested_bounds_from_diagnostic,
    format_multiplier_sensitivity_summary,
    run_scalar_matrix_sensitivity,
    save_multiplier_sensitivity_outputs,
    timestamped_sensitivity_output_dir,
)
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_combined_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim


## System And Data Setup

In [3]:
# Build the plant, load the canonical data bundle, and prepare the supervisory setpoint scenario.
SYS = NB["system_setup"]
system_params = SYS["system_params"].copy()
system_design_params = SYS["design_params"].copy()
system_steady_state_inputs = SYS["ss_inputs"].copy()
delta_t = SYS["delta_t_hours"]
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr_ss.ss_inputs, "y_ss": cstr_ss.y_ss}
setpoint_y = SYS["setpoint_range_phys"].copy()
u_min = SYS["input_bounds"]["u_min"].copy()
u_max = SYS["input_bounds"]["u_max"].copy()
system_data = load_polymer_system_data(REPO_ROOT, steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max, n_inputs=2, data_override=POLYMER_DATA_DIR_OVERRIDE)
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario_phys = SYS["rl_setpoints_phys"].copy()
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])


## Run / Reward / Agent Setup

In [4]:
# Run-profile, controller, reward, and agent setup.
EPISODE_CFG = NB["episode_defaults"]
CTRL = NB["controller"]
if HORIZON_AGENT_KIND not in {"dqn", "dueling_dqn"}:
    raise ValueError("HORIZON_AGENT_KIND must be 'dqn' or 'dueling_dqn'.")
HORIZON_CFG = NB["horizon_dueling_agent"] if HORIZON_AGENT_KIND == "dueling_dqn" else NB["horizon_agent"]
MATRIX_TD3_CFG = NB.get("matrix_td3_agent", NB["td3_agent"])
MATRIX_SAC_CFG = NB.get("matrix_sac_agent", NB["sac_agent"])
WEIGHTS_TD3_CFG = NB.get("weights_td3_agent", NB["td3_agent"])
WEIGHTS_SAC_CFG = NB.get("weights_sac_agent", NB["sac_agent"])
RESIDUAL_TD3_CFG = NB.get("residual_td3_agent", NB["td3_agent"])
RESIDUAL_SAC_CFG = NB.get("residual_sac_agent", NB["sac_agent"])
# Backward-compatible aliases used by summary/debug variables below.
TD3_CFG = MATRIX_TD3_CFG
SAC_CFG = MATRIX_SAC_CFG
REWARD_CFG = NB["reward"]

n_tests = int(EPISODE_CFG["n_tests"] if N_TESTS_OVERRIDE is None else N_TESTS_OVERRIDE)
set_points_len = int(EPISODE_CFG["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else SET_POINTS_LEN_OVERRIDE)
warm_start = int(EPISODE_CFG["warm_start"] if WARM_START_OVERRIDE is None else WARM_START_OVERRIDE)
TEST_CYCLE = list(EPISODE_CFG["test_cycle"] if TEST_CYCLE_OVERRIDE is None else TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = int(RUN_PROFILE["plot_start_episode"] if PLOT_START_EPISODE_OVERRIDE is None else PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = int(RUN_PROFILE["compare_start_episode"] if COMPARE_START_EPISODE_OVERRIDE is None else COMPARE_START_EPISODE_OVERRIDE)
ACTIVE_AGENT_SUFFIX = build_combined_suffix()
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or RUN_PROFILE["result_prefix_template"].format(suffix=ACTIVE_AGENT_SUFFIX)
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or RUN_PROFILE["compare_prefix_template"].format(suffix=ACTIVE_AGENT_SUFFIX)
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, data_override=POLYMER_DATA_DIR_OVERRIDE)
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
MISMATCH_CLIP = CTRL["mismatch_clip"]
INNOVATION_SCALE_MODE = CTRL["innovation_scale_mode"]
INNOVATION_SCALE_REF = CTRL["innovation_scale_ref"]
TRACKING_SCALE_MODE = CTRL["tracking_scale_mode"]
TRACKING_ETA_TOL = CTRL["tracking_eta_tol"]
TRACKING_SCALE_FLOOR = CTRL["tracking_scale_floor"]
TRACKING_SCALE_FLOOR_MODE = CTRL["tracking_scale_floor_mode"]
BASE_STATE_NORM_MODE = CTRL["base_state_norm_mode"]
BASE_STATE_RUNNING_NORM_CLIP = CTRL["base_state_running_norm_clip"]
BASE_STATE_RUNNING_NORM_EPS = CTRL["base_state_running_norm_eps"]
MISMATCH_FEATURE_TRANSFORM_MODE = CTRL["mismatch_feature_transform_mode"]
MISMATCH_TRANSFORM_TANH_SCALE = CTRL["mismatch_transform_tanh_scale"]
MISMATCH_TRANSFORM_POST_CLIP = CTRL["mismatch_transform_post_clip"]
OBSERVER_UPDATE_ALIGNMENT = CTRL["observer_update_alignment"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HORIZON_EXPLORATION_MODE = HORIZON_CFG["exploration_mode"]
HORIZON_LOSS_TYPE = HORIZON_CFG["loss_type"]
HORIZON_BUFFER_SIZE = int(HORIZON_CFG["buffer_size"])
HORIZON_REPLAY_FRAC_PER = float(HORIZON_CFG["replay_frac_per"])
HORIZON_REPLAY_FRAC_RECENT = float(HORIZON_CFG["replay_frac_recent"])
HORIZON_REPLAY_RECENT_WINDOW_MULT = int(HORIZON_CFG["replay_recent_window_mult"])
HORIZON_REPLAY_RECENT_WINDOW = int(HORIZON_CFG["replay_recent_window"]) if HORIZON_CFG["replay_recent_window"] is not None else min(HORIZON_BUFFER_SIZE, HORIZON_REPLAY_RECENT_WINDOW_MULT * set_points_len)
HORIZON_REPLAY_ALPHA = float(HORIZON_CFG["replay_alpha"])
HORIZON_REPLAY_BETA_START = float(HORIZON_CFG["replay_beta_start"])
HORIZON_REPLAY_BETA_END = float(HORIZON_CFG["replay_beta_end"])
HORIZON_REPLAY_BETA_STEPS = int(HORIZON_CFG["replay_beta_steps"])
TD3_EXPLORATION_MODE = TD3_CFG["exploration_mode"]
TD3_LOSS_TYPE = TD3_CFG["loss_type"]
SAC_LOSS_TYPE = SAC_CFG["loss_type"]
USE_SHIFTED_MPC_WARM_START = CTRL["use_shifted_mpc_warm_start"]
DECISION_INTERVAL = int(CTRL["decision_interval"])
TD3_PARAM_NOISE_RESAMPLE_INTERVAL = TD3_CFG["param_noise_resample_interval"]
TD3_BUFFER_SIZE = int(TD3_CFG["buffer_size"])
TD3_REPLAY_FRAC_PER = float(TD3_CFG["replay_frac_per"])
TD3_REPLAY_FRAC_RECENT = float(TD3_CFG["replay_frac_recent"])
TD3_REPLAY_RECENT_WINDOW_MULT = int(TD3_CFG["replay_recent_window_mult"])
TD3_REPLAY_RECENT_WINDOW = int(TD3_CFG["replay_recent_window"]) if TD3_CFG["replay_recent_window"] is not None else min(TD3_BUFFER_SIZE, TD3_REPLAY_RECENT_WINDOW_MULT * set_points_len)
TD3_REPLAY_ALPHA = float(TD3_CFG["replay_alpha"])
TD3_REPLAY_BETA_START = float(TD3_CFG["replay_beta_start"])
TD3_REPLAY_BETA_END = float(TD3_CFG["replay_beta_end"])
TD3_REPLAY_BETA_STEPS = int(TD3_CFG["replay_beta_steps"])
SAC_BUFFER_SIZE = int(SAC_CFG["buffer_size"])
SAC_REPLAY_FRAC_PER = float(SAC_CFG["replay_frac_per"])
SAC_REPLAY_FRAC_RECENT = float(SAC_CFG["replay_frac_recent"])
SAC_REPLAY_RECENT_WINDOW_MULT = int(SAC_CFG["replay_recent_window_mult"])
SAC_REPLAY_RECENT_WINDOW = int(SAC_CFG["replay_recent_window"]) if SAC_CFG["replay_recent_window"] is not None else min(SAC_BUFFER_SIZE, SAC_REPLAY_RECENT_WINDOW_MULT * set_points_len)
SAC_REPLAY_ALPHA = float(SAC_CFG["replay_alpha"])
SAC_REPLAY_BETA_START = float(SAC_CFG["replay_beta_start"])
SAC_REPLAY_BETA_END = float(SAC_CFG["replay_beta_end"])
SAC_REPLAY_BETA_STEPS = int(SAC_CFG["replay_beta_steps"])
PREDICT_GRID = list(CTRL["predict_grid"])
CONTROL_GRID = list(CTRL["control_grid"])
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
predict_h = CTRL["predict_h"]
cont_h = CTRL["cont_h"]
Q1_penalty = CTRL["Q1_penalty"]
Q2_penalty = CTRL["Q2_penalty"]
R1_penalty = CTRL["R1_penalty"]
R2_penalty = CTRL["R2_penalty"]
MODEL_LOW = CTRL["model_low"].copy()
MODEL_HIGH = CTRL["model_high"].copy()
WEIGHTS_LOW = CTRL["weights_low"].copy()
WEIGHTS_HIGH = CTRL["weights_high"].copy()
RESIDUAL_LOW = CTRL["residual_low"].copy()
RESIDUAL_HIGH = CTRL["residual_high"].copy()
nominal_qs = CTRL["nominal_qs"]
nominal_qi = CTRL["nominal_qi"]
nominal_hA = CTRL["nominal_ha"]
qi_change = CTRL["qi_change"]
qs_change = CTRL["qs_change"]
ha_change = CTRL["ha_change"]
REPLAY_SETTINGS = {
    "horizon": {
        "buffer_size": HORIZON_BUFFER_SIZE,
        "replay_frac_per": HORIZON_REPLAY_FRAC_PER,
        "replay_frac_recent": HORIZON_REPLAY_FRAC_RECENT,
        "replay_recent_window_mult": HORIZON_REPLAY_RECENT_WINDOW_MULT,
        "replay_recent_window": HORIZON_REPLAY_RECENT_WINDOW,
        "replay_alpha": HORIZON_REPLAY_ALPHA,
        "replay_beta_start": HORIZON_REPLAY_BETA_START,
        "replay_beta_end": HORIZON_REPLAY_BETA_END,
        "replay_beta_steps": HORIZON_REPLAY_BETA_STEPS,
    },
    "td3": {
        "buffer_size": TD3_BUFFER_SIZE,
        "replay_frac_per": TD3_REPLAY_FRAC_PER,
        "replay_frac_recent": TD3_REPLAY_FRAC_RECENT,
        "replay_recent_window_mult": TD3_REPLAY_RECENT_WINDOW_MULT,
        "replay_recent_window": TD3_REPLAY_RECENT_WINDOW,
        "replay_alpha": TD3_REPLAY_ALPHA,
        "replay_beta_start": TD3_REPLAY_BETA_START,
        "replay_beta_end": TD3_REPLAY_BETA_END,
        "replay_beta_steps": TD3_REPLAY_BETA_STEPS,
    },
    "sac": {
        "buffer_size": SAC_BUFFER_SIZE,
        "replay_frac_per": SAC_REPLAY_FRAC_PER,
        "replay_frac_recent": SAC_REPLAY_FRAC_RECENT,
        "replay_recent_window_mult": SAC_REPLAY_RECENT_WINDOW_MULT,
        "replay_recent_window": SAC_REPLAY_RECENT_WINDOW,
        "replay_alpha": SAC_REPLAY_ALPHA,
        "replay_beta_start": SAC_REPLAY_BETA_START,
        "replay_beta_end": SAC_REPLAY_BETA_END,
        "replay_beta_steps": SAC_REPLAY_BETA_STEPS,
    },
}
MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug, Q_out=np.array([Q1_penalty, Q2_penalty], float), R_in=np.array([R1_penalty, R2_penalty], float), NP=predict_h, NC=cont_h)
# Reward setup.
reward_params, reward_fn = make_reward_fn_relative_QR(data_min, data_max, N_INPUTS, **REWARD_CFG)

def _replay_params(cfg):
    buffer_size = int(cfg["buffer_size"])
    replay_recent_window_mult = int(cfg["replay_recent_window_mult"])
    replay_recent_window = (
        int(cfg["replay_recent_window"])
        if cfg["replay_recent_window"] is not None
        else min(buffer_size, replay_recent_window_mult * set_points_len)
    )
    return {
        "buffer_size": buffer_size,
        "replay_frac_per": float(cfg["replay_frac_per"]),
        "replay_frac_recent": float(cfg["replay_frac_recent"]),
        "replay_recent_window_mult": replay_recent_window_mult,
        "replay_recent_window": replay_recent_window,
        "replay_alpha": float(cfg["replay_alpha"]),
        "replay_beta_start": float(cfg["replay_beta_start"]),
        "replay_beta_end": float(cfg["replay_beta_end"]),
        "replay_beta_steps": int(cfg["replay_beta_steps"]),
    }


def make_continuous_agent(agent_kind, state_dim, action_dim, td3_cfg, sac_cfg):
    if agent_kind == "td3":
        replay = _replay_params(td3_cfg)
        return TD3Agent(state_dim=state_dim, action_dim=action_dim, actor_hidden=list(td3_cfg["actor_hidden"]), critic_hidden=list(td3_cfg["critic_hidden"]), gamma=td3_cfg["gamma"], actor_lr=td3_cfg["actor_lr"], critic_lr=td3_cfg["critic_lr"], batch_size=td3_cfg["batch_size"], n_step=td3_cfg["n_step"], multistep_mode=td3_cfg["multistep_mode"], lambda_value=td3_cfg["lambda_value"], grad_clip_norm=td3_cfg.get("grad_clip_norm", 10.0), policy_delay=td3_cfg["policy_delay"], target_policy_smoothing_noise_std=td3_cfg["target_policy_smoothing_noise_std"], noise_clip=td3_cfg["noise_clip"], max_action=td3_cfg["max_action"], tau=td3_cfg["tau"], std_start=td3_cfg["std_start"], std_end=td3_cfg["std_end"], std_decay_rate=td3_cfg["std_decay_rate"], std_decay_mode=td3_cfg["std_decay_mode"], buffer_size=replay["buffer_size"], replay_frac_per=replay["replay_frac_per"], replay_frac_recent=replay["replay_frac_recent"], replay_recent_window=replay["replay_recent_window"], replay_alpha=replay["replay_alpha"], replay_beta_start=replay["replay_beta_start"], replay_beta_end=replay["replay_beta_end"], replay_beta_steps=replay["replay_beta_steps"], device=DEVICE, actor_freeze=td3_cfg["actor_freeze"], exploration_mode=td3_cfg["exploration_mode"], loss_type=td3_cfg["loss_type"], param_noise_resample_interval=td3_cfg["param_noise_resample_interval"])
    if agent_kind == "sac":
        replay = _replay_params(sac_cfg)
        target_entropy = -action_dim if sac_cfg["target_entropy"] == "auto_negative_action_dim" else sac_cfg["target_entropy"]
        return SACAgent(state_dim=state_dim, action_dim=action_dim, actor_hidden=list(sac_cfg["actor_hidden"]), critic_hidden=list(sac_cfg["critic_hidden"]), gamma=sac_cfg["gamma"], actor_lr=sac_cfg["actor_lr"], critic_lr=sac_cfg["critic_lr"], alpha_lr=sac_cfg["alpha_lr"], batch_size=sac_cfg["batch_size"], n_step=sac_cfg["n_step"], multistep_mode=sac_cfg["multistep_mode"], lambda_value=sac_cfg["lambda_value"], grad_clip_norm=sac_cfg["grad_clip_norm"], init_alpha=sac_cfg["init_alpha"], learn_alpha=sac_cfg["learn_alpha"], target_entropy=target_entropy, target_update=sac_cfg["target_update"], tau=sac_cfg["tau"], hard_update_interval=sac_cfg["hard_update_interval"], activation=sac_cfg["activation"], use_layernorm=sac_cfg["use_layernorm"], dropout=sac_cfg["dropout"], max_action=sac_cfg["max_action"], buffer_size=replay["buffer_size"], replay_frac_per=replay["replay_frac_per"], replay_frac_recent=replay["replay_frac_recent"], replay_recent_window=replay["replay_recent_window"], replay_alpha=replay["replay_alpha"], replay_beta_start=replay["replay_beta_start"], replay_beta_end=replay["replay_beta_end"], replay_beta_steps=replay["replay_beta_steps"], device=DEVICE, use_adamw=sac_cfg["use_adamw"], actor_freeze=sac_cfg["actor_freeze"], loss_type=sac_cfg["loss_type"])
    raise ValueError("Continuous agent kind must be 'td3' or 'sac'.")
horizon_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, HORIZON_STATE_MODE)
matrix_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, MATRIX_STATE_MODE)
weights_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, WEIGHTS_STATE_MODE)
residual_state_dim = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, RESIDUAL_STATE_MODE, append_rho_to_state=(RESIDUAL_STATE_MODE == "mismatch" and APPEND_RHO_TO_STATE))

# Agent setup.
horizon_agent_kwargs = dict(
    state_dim=horizon_state_dim,
    action_dim=len(HORIZON_RECIPES),
    hidden_dim=list(HORIZON_CFG["hidden_layers"]),
    gamma=HORIZON_CFG["gamma"],
    lr=HORIZON_CFG["lr"],
    batch_size=HORIZON_CFG["batch_size"],
    buffer_size=HORIZON_BUFFER_SIZE,
    replay_frac_per=HORIZON_REPLAY_FRAC_PER,
    replay_frac_recent=HORIZON_REPLAY_FRAC_RECENT,
    replay_recent_window=HORIZON_REPLAY_RECENT_WINDOW,
    replay_alpha=HORIZON_REPLAY_ALPHA,
    replay_beta_start=HORIZON_REPLAY_BETA_START,
    replay_beta_end=HORIZON_REPLAY_BETA_END,
    replay_beta_steps=HORIZON_REPLAY_BETA_STEPS,
    n_step=HORIZON_CFG["n_step"],
    multistep_mode=HORIZON_CFG["multistep_mode"],
    lambda_value=HORIZON_CFG["lambda_value"],
    grad_clip_norm=HORIZON_CFG["grad_clip_norm"],
    double_dqn=HORIZON_CFG["double_dqn"],
    target_update=HORIZON_CFG["target_update"],
    tau=HORIZON_CFG["tau"],
    hard_update_interval=HORIZON_CFG["hard_update_interval"],
    activation=HORIZON_CFG["activation"],
    use_layer_norm=HORIZON_CFG["use_layer_norm"],
    dropout=HORIZON_CFG["dropout"],
    device=DEVICE,
    exploration_mode=HORIZON_EXPLORATION_MODE,
    loss_type=HORIZON_LOSS_TYPE,
    eps_start=HORIZON_CFG["eps_start"],
    eps_end=HORIZON_CFG["eps_end"],
    eps_decay_rate=HORIZON_CFG["eps_decay_rate"],
    eps_decay_steps=HORIZON_CFG.get("eps_decay_steps", 100_000),
    eps_decay_mode=HORIZON_CFG["eps_decay_mode"],
)
if HORIZON_AGENT_KIND == "dqn":
    horizon_agent = DQNAgent(**horizon_agent_kwargs, target_combine=HORIZON_CFG.get("target_combine"))
elif HORIZON_AGENT_KIND == "dueling_dqn":
    horizon_agent = DuelingDQNAgent(**horizon_agent_kwargs)
else:
    raise ValueError("HORIZON_AGENT_KIND must be 'dqn' or 'dueling_dqn'.")
matrix_agent = make_continuous_agent(MATRIX_AGENT_KIND, matrix_state_dim, 1 + N_INPUTS, MATRIX_TD3_CFG, MATRIX_SAC_CFG)
weights_agent = make_continuous_agent(WEIGHTS_AGENT_KIND, weights_state_dim, 4, WEIGHTS_TD3_CFG, WEIGHTS_SAC_CFG)
residual_agent = make_continuous_agent(RESIDUAL_AGENT_KIND, residual_state_dim, N_INPUTS, RESIDUAL_TD3_CFG, RESIDUAL_SAC_CFG)


## Resolved Summary

In [5]:
print_grouped_notebook_summary(
    "Polymer combined run summary",
    {
        "Paths": {"Repo root": REPO_ROOT, "Data dir": DATA_DIR, "Results dir": RESULT_DIR, "Baseline MPC": BASELINE_MPC_PATH},
        "Run setup": {"Run mode": RUN_MODE, "n_tests": n_tests, "set_points_len": set_points_len, "warm_start": warm_start, "test_cycle": TEST_CYCLE, "decision_interval": DECISION_INTERVAL, "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START, "matrix_estimator_mode": "fixed_nominal"},
        "System / controller": {"predict_h": predict_h, "cont_h": cont_h, "observer_poles": POLYMER_OBSERVER_POLES.tolist(), "predict_grid": PREDICT_GRID, "control_grid": CONTROL_GRID},
        "Reward": reward_params,
        "Agents": {"horizon": f"enabled={ENABLE_HORIZON}, kind={HORIZON_AGENT_KIND}, state_mode={HORIZON_STATE_MODE}", "matrix": f"enabled={ENABLE_MATRIX}, kind={MATRIX_AGENT_KIND}, state_mode={MATRIX_STATE_MODE}", "weights": f"enabled={ENABLE_WEIGHTS}, kind={WEIGHTS_AGENT_KIND}, state_mode={WEIGHTS_STATE_MODE}", "residual": f"enabled={ENABLE_RESIDUAL}, kind={RESIDUAL_AGENT_KIND}, state_mode={RESIDUAL_STATE_MODE}, rho={USE_RHO_AUTHORITY}"},
        "Replay": REPLAY_SETTINGS,
        "Mismatch": {"clip": MISMATCH_CLIP, "innovation_scale_mode": INNOVATION_SCALE_MODE, "tracking_scale_mode": TRACKING_SCALE_MODE, "tracking_eta_tol": TRACKING_ETA_TOL, "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE},
        "Residual authority": {"use_rho": USE_RHO_AUTHORITY, "append_rho_to_state": APPEND_RHO_TO_STATE, "rho_floor": AUTHORITY_RHO_FLOOR, "rho_power": AUTHORITY_RHO_POWER},
        "Plotting / export": {"style_profile": STYLE_PROFILE, "save_pdf": SAVE_PDF, "result_prefix": RESULT_PREFIX, "compare_prefix": COMPARE_PREFIX, "plot_start_episode": PLOT_START_EPISODE, "compare_start_episode": COMPARE_START_EPISODE},
    },
)

Polymer combined run summary

[Paths]
  Repo root           : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer
  Data dir            : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Data
  Results dir         : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results
  Baseline MPC        : C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Data\mpc_results_dist.pickle

[Run setup]
  Run mode            : disturb
  n_tests             : 200
  set_points_len      : 400
  warm_start          : 10
  test_cycle          : [False, False, False, False, False]
  decision_interval   : 4
  use_shifted_mpc_warm_start: False
  matrix_estimator_mode: fixed_nominal

[System / controller]
  predict_h           : 9
  cont_h              : 3
  observer_poles      : [0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966, 0.42900673, 0.4228262, 0.96916776, 0.91230187]
  predict_grid        : [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  control_grid        : [3, 4, 5, 6, 7, 8,

In [6]:
# Offline rho + finite-horizon gain sensitivity diagnostic for the scalar matrix block.
OFFLINE_MULTIPLIER_DIAGNOSTICS = dict(CTRL.get("offline_multiplier_diagnostics", {}))
offline_multiplier_diagnostic_result = None
offline_multiplier_diagnostic_paths = {}

if ENABLE_MATRIX and bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("enabled", False)):
    if bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("apply_suggested_caps", False)):
        print("Step 1 diagnostic is advisory only; apply_suggested_caps is ignored.")
    offline_multiplier_diagnostic_result = run_scalar_matrix_sensitivity(
        A_aug=A_aug,
        B_aug=B_aug,
        C_aug=C_aug,
        low_bounds=MODEL_LOW,
        high_bounds=MODEL_HIGH,
        predict_h=predict_h,
        n_outputs=N_OUTPUTS,
        epsilon_log=float(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("epsilon_log", 0.02)),
        n_random_samples=int(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("n_random_samples", 2_000)),
        seed=int(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("seed", 42)),
        rho_target=float(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("rho_target", 0.995)),
        gain_threshold=float(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("gain_threshold", 0.25)),
        system_name="polymer",
        method_family="combined_matrix",
    )
    if bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("save_outputs", True)):
        offline_multiplier_diagnostic_dir = timestamped_sensitivity_output_dir(
            RESULT_DIR,
            system_name="polymer",
            method_family="combined_matrix",
            agent_kind=MATRIX_AGENT_KIND,
            run_mode=RUN_MODE,
        )
        offline_multiplier_diagnostic_paths = save_multiplier_sensitivity_outputs(
            offline_multiplier_diagnostic_result,
            offline_multiplier_diagnostic_dir,
            make_plots=bool(OFFLINE_MULTIPLIER_DIAGNOSTICS.get("make_plots", True)),
        )
        print(f"Offline multiplier diagnostic outputs: {offline_multiplier_diagnostic_dir}")
    print(format_multiplier_sensitivity_summary(offline_multiplier_diagnostic_result))
elif ENABLE_MATRIX:
    print("Offline multiplier sensitivity diagnostic disabled for the combined matrix block.")

# Step 2 release-protected advisory caps. Diagnostic bounds are used only as an execution guard.
RELEASE_PROTECTED_ADVISORY_CAPS = dict(CTRL.get("release_protected_advisory_caps", {}))
if ENABLE_MATRIX and bool(RELEASE_PROTECTED_ADVISORY_CAPS.get("enabled", False)):
    if offline_multiplier_diagnostic_result is None:
        raise RuntimeError(
            "Step 2 release-protected advisory caps are enabled, but Step 1 offline multiplier diagnostic did not run."
        )
    RELEASE_PROTECTED_ADVISORY_CAPS["advisory_bounds"] = extract_suggested_bounds_from_diagnostic(
        offline_multiplier_diagnostic_result,
        labels=["alpha"] + [f"B_col_{idx + 1}" for idx in range(B_aug.shape[1])],
    )
    print("Step 2 release-protected advisory caps enabled for the combined matrix block.")
elif ENABLE_MATRIX:
    print("Step 2 release-protected advisory caps disabled for the combined matrix block.")
else:
    RELEASE_PROTECTED_ADVISORY_CAPS["enabled"] = False


Offline multiplier diagnostic outputs: C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results\offline_multiplier_sensitivity\polymer_combined_matrix_td3_disturb_20260426_140731
Offline multiplier sensitivity diagnostic
  system: polymer
  method: combined_matrix
  nominal rho: 0.946394
  nominal gain drift: 0
  sampled candidates: 2000
  unstable fraction: 0.0000
  near-unit fraction: 0.0060
  gain ratio p95/max: 0.7344 / 0.7855
  most sensitive coordinates:
    alpha (A_scalar): S_rho=0.9465, S_G=4.157, rho-=0.927654, rho+=0.965512, G-=0.07381, G+=0.08308
    B_col_2 (B_col): S_rho=0, S_G=0.8935, rho-=0.946394, rho+=0.946394, G-=0.01742, G+=0.01777
    B_col_1 (B_col): S_rho=0, S_G=0.4775, rho-=0.946394, rho+=0.946394, G-=0.009413, G+=0.009603
Step 2 release-protected advisory caps enabled for the combined matrix block.


## Run

In [7]:
# Assemble the shared runner configuration and execute the rollout.
combined_cfg = {
    "run_mode": RUN_MODE,
    "horizon_agent_kind": HORIZON_AGENT_KIND,
    "notebook_source": "RL_assisted_MPC_combined_unified.ipynb",
    "decision_interval": DECISION_INTERVAL,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "td3_post_warm_start_action_freeze_subepisodes": int(NB["td3_post_warm_start_action_freeze_subepisodes"]),
    "td3_post_warm_start_actor_freeze_subepisodes": int(NB["td3_post_warm_start_actor_freeze_subepisodes"]),
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "horizon_cfg": {
        "enabled": ENABLE_HORIZON,
        "agent_kind": HORIZON_AGENT_KIND,
        "state_mode": HORIZON_STATE_MODE,
                "mismatch_clip": MISMATCH_CLIP,
    "innovation_scale_mode": INNOVATION_SCALE_MODE,
    "innovation_scale_ref": INNOVATION_SCALE_REF,
    "tracking_scale_mode": TRACKING_SCALE_MODE,
    "tracking_eta_tol": TRACKING_ETA_TOL,
    "tracking_scale_floor": TRACKING_SCALE_FLOOR,
    "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE,
    "base_state_norm_mode": BASE_STATE_NORM_MODE,
    "base_state_running_norm_clip": BASE_STATE_RUNNING_NORM_CLIP,
    "base_state_running_norm_eps": BASE_STATE_RUNNING_NORM_EPS,
    "mismatch_feature_transform_mode": MISMATCH_FEATURE_TRANSFORM_MODE,
    "mismatch_transform_tanh_scale": MISMATCH_TRANSFORM_TANH_SCALE,
    "mismatch_transform_post_clip": MISMATCH_TRANSFORM_POST_CLIP,
    "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT,
        "horizon_recipes": HORIZON_RECIPES,
        "default_horizons": (predict_h, cont_h),
    },
    "matrix_cfg": {
        "enabled": ENABLE_MATRIX,
        "agent_kind": MATRIX_AGENT_KIND,
        "state_mode": MATRIX_STATE_MODE,
                "mismatch_clip": MISMATCH_CLIP,
    "innovation_scale_mode": INNOVATION_SCALE_MODE,
    "innovation_scale_ref": INNOVATION_SCALE_REF,
    "tracking_scale_mode": TRACKING_SCALE_MODE,
    "tracking_eta_tol": TRACKING_ETA_TOL,
    "tracking_scale_floor": TRACKING_SCALE_FLOOR,
    "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE,
    "base_state_norm_mode": BASE_STATE_NORM_MODE,
    "base_state_running_norm_clip": BASE_STATE_RUNNING_NORM_CLIP,
    "base_state_running_norm_eps": BASE_STATE_RUNNING_NORM_EPS,
    "mismatch_feature_transform_mode": MISMATCH_FEATURE_TRANSFORM_MODE,
    "mismatch_transform_tanh_scale": MISMATCH_TRANSFORM_TANH_SCALE,
    "mismatch_transform_post_clip": MISMATCH_TRANSFORM_POST_CLIP,
    "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT,
        "low_coef": MODEL_LOW,
        "high_coef": MODEL_HIGH,
        "release_protected_advisory_caps": RELEASE_PROTECTED_ADVISORY_CAPS,
    },
    "weight_cfg": {
        "enabled": ENABLE_WEIGHTS,
        "agent_kind": WEIGHTS_AGENT_KIND,
        "state_mode": WEIGHTS_STATE_MODE,
                "mismatch_clip": MISMATCH_CLIP,
    "innovation_scale_mode": INNOVATION_SCALE_MODE,
    "innovation_scale_ref": INNOVATION_SCALE_REF,
    "tracking_scale_mode": TRACKING_SCALE_MODE,
    "tracking_eta_tol": TRACKING_ETA_TOL,
    "tracking_scale_floor": TRACKING_SCALE_FLOOR,
    "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE,
    "base_state_norm_mode": BASE_STATE_NORM_MODE,
    "base_state_running_norm_clip": BASE_STATE_RUNNING_NORM_CLIP,
    "base_state_running_norm_eps": BASE_STATE_RUNNING_NORM_EPS,
    "mismatch_feature_transform_mode": MISMATCH_FEATURE_TRANSFORM_MODE,
    "mismatch_transform_tanh_scale": MISMATCH_TRANSFORM_TANH_SCALE,
    "mismatch_transform_post_clip": MISMATCH_TRANSFORM_POST_CLIP,
    "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT,
        "low_coef": WEIGHTS_LOW,
        "high_coef": WEIGHTS_HIGH,
    },
    "residual_cfg": {
        "enabled": ENABLE_RESIDUAL,
        "agent_kind": RESIDUAL_AGENT_KIND,
        "state_mode": RESIDUAL_STATE_MODE,
                "mismatch_clip": MISMATCH_CLIP,
    "innovation_scale_mode": INNOVATION_SCALE_MODE,
    "innovation_scale_ref": INNOVATION_SCALE_REF,
    "tracking_scale_mode": TRACKING_SCALE_MODE,
    "tracking_eta_tol": TRACKING_ETA_TOL,
    "tracking_scale_floor": TRACKING_SCALE_FLOOR,
    "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE,
    "base_state_norm_mode": BASE_STATE_NORM_MODE,
    "base_state_running_norm_clip": BASE_STATE_RUNNING_NORM_CLIP,
    "base_state_running_norm_eps": BASE_STATE_RUNNING_NORM_EPS,
    "mismatch_feature_transform_mode": MISMATCH_FEATURE_TRANSFORM_MODE,
    "mismatch_transform_tanh_scale": MISMATCH_TRANSFORM_TANH_SCALE,
    "mismatch_transform_post_clip": MISMATCH_TRANSFORM_POST_CLIP,
    "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT,
        "authority_use_rho": USE_RHO_AUTHORITY,
    "use_rho_authority": USE_RHO_AUTHORITY,
    "append_rho_to_state": APPEND_RHO_TO_STATE,
    "authority_beta_res": AUTHORITY_BETA_RES,
    "authority_du0_res": AUTHORITY_DU0_RES,
    "authority_eta_tol": AUTHORITY_ETA_TOL,
    "authority_rho_floor": AUTHORITY_RHO_FLOOR,
    "authority_rho_power": AUTHORITY_RHO_POWER,
    "rho_mapping_mode": RHO_MAPPING_MODE,
    "authority_rho_k": AUTHORITY_RHO_K,
    "residual_zero_deadband_enabled": RESIDUAL_ZERO_DEADBAND_ENABLED,
    "residual_zero_tracking_raw_threshold": RESIDUAL_ZERO_TRACKING_RAW_THRESHOLD,
    "residual_zero_innovation_raw_threshold": RESIDUAL_ZERO_INNOVATION_RAW_THRESHOLD,
        "low_coef": RESIDUAL_LOW,
        "high_coef": RESIDUAL_HIGH,
    },
}

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
agents = {}
if ENABLE_HORIZON:
    agents["horizon"] = horizon_agent
if ENABLE_MATRIX:
    agents["matrix"] = matrix_agent
if ENABLE_WEIGHTS:
    agents["weights"] = weights_agent
if ENABLE_RESIDUAL:
    agents["residual"] = residual_agent

runtime_ctx = {
    "system": cstr,
    "agents": agents,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": POLYMER_OBSERVER_POLES.copy(),
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "reward_params": reward_params,
}

result_bundle = run_combined_supervisor(combined_cfg=combined_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH


Sub_Episode: 1 | avg. reward: -3.366625844380668 | Hp,Hc: (np.int64(9), np.int64(3)) | alpha: 1.0 | weights: [1. 1. 1. 1.] | residual: [-1.17574848e-08 -4.70503352e-08]
Sub_Episode: 2 | avg. reward: -4.417571695433463 | Hp,Hc: (np.int64(9), np.int64(3)) | alpha: 1.0 | weights: [1. 1. 1. 1.] | residual: [-2.31331523e-08  4.71642210e-08]
Sub_Episode: 3 | avg. reward: -4.417592476065279 | Hp,Hc: (np.int64(9), np.int64(3)) | alpha: 1.0 | weights: [1. 1. 1. 1.] | residual: [2.76633774e-08 4.51300934e-08]
Sub_Episode: 4 | avg. reward: -4.4173276842395515 | Hp,Hc: (np.int64(9), np.int64(3)) | alpha: 1.0 | weights: [1. 1. 1. 1.] | residual: [3.78852505e-09 3.70101141e-08]
Sub_Episode: 5 | avg. reward: -4.417496215574175 | Hp,Hc: (np.int64(9), np.int64(3)) | alpha: 1.0 | weights: [1. 1. 1. 1.] | residual: [-1.03549036e-08  2.65811124e-08]
Sub_Episode: 6 | avg. reward: -4.417020413157124 | Hp,Hc: (np.int64(9), np.int64(3)) | alpha: 1.0 | weights: [1. 1. 1. 1.] | residual: [9.89598714e-09 3.48943

## Plotting And Export

In [8]:
# Generate saved figures, then run the baseline MPC comparison explicitly.
out_dir_rl = plot_combined_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
        "system_metadata": POLYMER_SYSTEM_METADATA,
        "include_baseline_compare": False,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_PROFILE["compare_mode"],
    start_episode=COMPARE_START_EPISODE,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print("Combined result directory:", out_dir_rl)
print("Comparison result directory:", out_dir_cmp)


Combined result directory: C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results\combined_disturb_h_dqn_mismatch__m_td3_mismatch__w_td3_mismatch__r_td3_mismatch_rho\20260426_193310
Comparison result directory: C:\Users\HAMEDI\Desktop\RL_assisted_MPC_polymer\Polymer\Results\disturb_compare_combined_h_dqn_mismatch__m_td3_mismatch__w_td3_mismatch__r_td3_mismatch_rho\20260426_193431
